# 第2章 · 2.1 互联网与万维网 (The Internet & The World Wide Web)
> Cambridge International AS & A Level Computer Science (9618)

---

本 notebook 将从零开始介绍：
1. Internet vs WWW — 它们不是同一回事！
2. Internet 简史
3. IP 地址（IPv4 和 IPv6）
4. URL 结构解析
5. DNS 域名系统
6. 网页加载全过程
7. Ethernet 和 CSMA/CD
8. Cloud Computing 云计算
9. Bit Streaming 比特流
10. Internet 硬件
11. 练习题

## 1. Internet vs WWW — 它们不是同一回事！

这是很多人搞混的概念，让我们彻底搞清楚：

### 1.1 Internet（互联网）

**Internet** 是一个 **全球性的计算机网络基础设施**——它是由无数个网络互相连接形成的"网络的网络"。

> **类比**：Internet 就像 **公路系统**——它是道路和基础设施本身。

**Internet 提供的是：**
- 物理连接（光纤、铜缆、卫星等）
- 通信协议（TCP/IP）
- 数据传输的能力

### 1.2 WWW (World Wide Web)（万维网）

**WWW** 是运行在 Internet **之上** 的一个 **服务/应用**。它是由 **网页 (Web Pages)** 通过 **超链接 (Hyperlinks)** 连接在一起的信息系统。

> **类比**：WWW 就像在公路上行驶的 **快递卡车**——它利用公路（Internet）来传递包裹（网页）。

**WWW 的组成：**
- HTML 网页
- HTTP/HTTPS 协议
- 浏览器（客户端）
- Web 服务器

### 1.3 关键区别

| | Internet | WWW |
|--|---------|-----|
| 是什么 | 全球计算机网络基础设施 | 运行在 Internet 上的服务 |
| 发明时间 | 1969年（ARPANET） | 1989年（Tim Berners-Lee） |
| 使用的协议 | TCP/IP | HTTP/HTTPS |
| 包含 | 一切网络通信 | 网页和超链接 |
| 关系 | 基础 | WWW 是 Internet 的一部分 |

### 1.4 Internet 上不只有 WWW！

Internet 上运行的其他服务：
- **Email**（电子邮件）— 使用 SMTP/POP3/IMAP 协议
- **FTP**（文件传输）— 使用 FTP 协议
- **VoIP**（网络电话）— 如 Skype、微信语音
- **Instant Messaging**（即时通信）— 如 WhatsApp
- **Online Gaming**（在线游戏）
- **Video Streaming**（视频流）— 如 Netflix、B站

> **结论**：WWW ⊂ Internet（WWW 是 Internet 的子集）

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

# 可视化：Internet vs WWW 的关系
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('Internet vs WWW: The Relationship', fontsize=14, fontweight='bold')
ax.set_aspect('equal')
ax.axis('off')

# Internet (large circle)
internet_circle = plt.Circle((5, 5), 4.2, color='#E3F2FD', ec='#1565C0', lw=3)
ax.add_patch(internet_circle)
ax.text(5, 9.0, 'Internet', fontsize=16, fontweight='bold', color='#1565C0', ha='center')
ax.text(5, 8.5, '(Global network infrastructure)', fontsize=10, color='#1565C0', ha='center')

# WWW (medium circle inside)
www_circle = plt.Circle((4, 5), 2, color='#C8E6C9', ec='#2E7D32', lw=2.5)
ax.add_patch(www_circle)
ax.text(4, 5.3, 'WWW', fontsize=14, fontweight='bold', color='#2E7D32', ha='center')
ax.text(4, 4.7, 'Web Pages\nHTTP/HTTPS', fontsize=9, color='#2E7D32', ha='center')

# Other services (small circles)
services = [
    (7.5, 7, 'Email', '#FF7043'),
    (7.8, 5, 'FTP', '#AB47BC'),
    (7.2, 3, 'VoIP', '#26C6DA'),
    (5.5, 1.5, 'Gaming', '#FF9800'),
    (3, 1.5, 'Streaming', '#78909C'),
]

for x, y, name, c in services:
    circle = plt.Circle((x, y), 0.6, color=c, ec='black', lw=1.5, alpha=0.8)
    ax.add_patch(circle)
    ax.text(x, y, name, fontsize=8, fontweight='bold', color='white', ha='center', va='center')

ax.text(5, 0.3, 'WWW is just ONE of many services running on the Internet!',
        ha='center', fontsize=11, style='italic', color='#333')

plt.tight_layout()
plt.savefig('internet_vs_www.png', dpi=100, bbox_inches='tight')
plt.show()


## 2. Internet 简史

| 年份 | 事件 |
|------|------|
| 1969 | **ARPANET** 诞生 — 美国国防部的研究网络，最初只连接4所大学 |
| 1971 | 第一封 **Email** 被发送 |
| 1974 | **TCP/IP** 协议被提出 — Internet 的基础语言 |
| 1983 | ARPANET 正式采用 TCP/IP — 被视为 "Internet 的生日" |
| 1989 | Tim Berners-Lee 在 CERN 发明 **WWW** |
| 1990 | 第一个 **Web 浏览器** 和 **Web 服务器** 诞生 |
| 1993 | Mosaic 浏览器发布 — 让普通人也能上网 |
| 1995 | 商业 Internet 兴起（Amazon、eBay）|
| 2004 | Web 2.0 时代（用户生成内容：Facebook、YouTube）|
| 2010s | 移动互联网时代（智能手机普及）|
| 2020s | 5G、物联网 (IoT)、云计算 |

> **核心启示**：Internet (1969) 比 WWW (1989) 早了整整 **20年**！它们不是同一个东西。

## 3. IP 地址 (IP Addresses)

### 3.1 什么是 IP 地址？

**IP 地址 (Internet Protocol Address)** 是网络中每台设备的 **唯一标识符**——就像你家的门牌号。

> **类比**：如果 MAC 地址是你的"身份证号"（硬件固定），那 IP 地址就是你的"家庭住址"（可以搬家改变）。

### 3.2 IPv4 地址

**格式**：`192.168.1.100`（4 组十进制数，每组 0-255，用点分隔）

**底层**：每组是 8 bits (1 byte)，总共 **32 bits**。

```
192     . 168     . 1       . 100
11000000  10101000  00000001  01100100
```

**IPv4 的问题**：32 bits 最多只能表示 2^32 = **42 亿** 个地址——全世界 80 亿人口，设备数量远超 42 亿，不够用了！

### 3.3 IPv6 地址

**格式**：`2001:0db8:85a3:0000:0000:8a2e:0370:7334`（8 组十六进制数，用冒号分隔）

**底层**：总共 **128 bits**。

2^128 = 约 3.4 × 10^38 个地址——足够给地球上每一粒沙子都分配一个 IP 地址！

### 3.4 公有 IP vs 私有 IP

| 类型 | 范围 | 用途 |
|------|------|------|
| 公有 IP | 全球唯一 | 在 Internet 上标识设备 |
| 私有 IP | 10.x.x.x / 172.16-31.x.x / 192.168.x.x | 在 LAN 内部使用 |

你家路由器有一个 **公有 IP**（面向 Internet），你家里的设备使用 **私有 IP**（面向 LAN）。路由器通过 **NAT (Network Address Translation)** 在两者之间转换。

In [ ]:
# IP 地址的二进制表示
def ip_to_binary(ip_str):
    # 将 IPv4 地址转换为二进制表示
    octets = ip_str.split('.')
    binary_parts = []
    for octet in octets:
        binary_parts.append(format(int(octet), '08b'))
    return '.'.join(binary_parts)

def binary_to_ip(binary_str):
    # 将二进制表示转换回 IPv4 地址
    parts = binary_str.replace('.', ' ').split()
    # Group into 8-bit blocks
    full_binary = ''.join(parts)
    octets = [full_binary[i:i+8] for i in range(0, 32, 8)]
    return '.'.join(str(int(octet, 2)) for octet in octets)

print("=" * 60)
print("  IPv4 Address Binary Representation")
print("=" * 60)

examples = ['192.168.1.100', '10.0.0.1', '255.255.255.0', '172.16.0.1', '8.8.8.8']

for ip in examples:
    binary = ip_to_binary(ip)
    print(f"\n  Decimal:  {ip:>15}")
    print(f"  Binary:   {binary}")

# IPv4 address space
print("\n" + "=" * 60)
print("  IPv4 vs IPv6 Address Space")
print("=" * 60)

ipv4_count = 2**32
ipv6_count = 2**128

print(f"\n  IPv4 (32 bits):  {ipv4_count:,} addresses")
print(f"                   = ~{ipv4_count/1e9:.1f} billion")
print(f"\n  IPv6 (128 bits): {ipv6_count:.2e} addresses")
print(f"                   = enough for every grain of sand on Earth!")

# Subnet example
print("\n" + "=" * 60)
print("  Subnet Mask Example")
print("=" * 60)

ip = '192.168.1.100'
mask = '255.255.255.0'
print(f"\n  IP Address:   {ip:>15}  =  {ip_to_binary(ip)}")
print(f"  Subnet Mask:  {mask:>15}  =  {ip_to_binary(mask)}")

# AND operation
ip_bits = ip_to_binary(ip).replace('.', '')
mask_bits = ip_to_binary(mask).replace('.', '')
network_bits = ''.join('1' if a == '1' and b == '1' else '0' for a, b in zip(ip_bits, mask_bits))
network = '.'.join(str(int(network_bits[i:i+8], 2)) for i in range(0, 32, 8))
print(f"  Network ID:   {network:>15}  (IP AND Mask)")
print(f"\n  -> This device is on the 192.168.1.x network")

## 4. URL 结构 (URL Structure)

### 什么是 URL？

**URL (Uniform Resource Locator)** = 统一资源定位符，就是你在浏览器地址栏输入的 **网址**。

### 拆解一个 URL：

```
https://www.example.com:443/path/to/page.html?key=value#section
  |       |        |    |      |                  |         |
Protocol  Sub    Domain Port   Path              Query    Fragment
(协议)  domain  (域名) (端口)  (路径)           (查询参数)  (锚点)
       (子域名)
```

| 部分 | 示例 | 说明 |
|------|------|------|
| Protocol | `https://` | 使用什么协议通信（HTTP 或 HTTPS） |
| Subdomain | `www` | 子域名（可选） |
| Domain | `example.com` | 主域名 |
| Port | `:443` | 端口号（HTTPS 默认 443，HTTP 默认 80） |
| Path | `/path/to/page.html` | 服务器上的文件路径 |
| Query | `?key=value` | 传递给服务器的参数 |
| Fragment | `#section` | 页面内的定位点 |

In [ ]:
# 解析 URL 的各个部分
from urllib.parse import urlparse, parse_qs

urls = [
    'https://www.google.com/search?q=computer+science&lang=en',
    'http://192.168.1.1:8080/admin/settings.html',
    'https://en.wikipedia.org/wiki/Computer_network#History',
    'ftp://files.example.com/documents/report.pdf',
]

print("=" * 70)
print("  URL Structure Analysis")
print("=" * 70)

for url in urls:
    parsed = urlparse(url)
    print(f"\n  Full URL: {url}")
    print(f"  {'Protocol:':<12} {parsed.scheme}")
    print(f"  {'Host:':<12} {parsed.hostname}")
    if parsed.port:
        print(f"  {'Port:':<12} {parsed.port}")
    print(f"  {'Path:':<12} {parsed.path}")
    if parsed.query:
        print(f"  {'Query:':<12} {parsed.query}")
        params = parse_qs(parsed.query)
        for k, v in params.items():
            print(f"    {k} = {v[0]}")
    if parsed.fragment:
        print(f"  {'Fragment:':<12} {parsed.fragment}")
    print("  " + "-" * 50)

## 5. DNS (Domain Name System) — 互联网的"电话簿"

### 5.1 为什么需要 DNS？

电脑用 **IP 地址**（如 `142.250.80.46`）来找到其他电脑，但人类记不住这些数字！

DNS 的作用就是把人类友好的 **域名**（如 `google.com`）翻译成机器需要的 **IP 地址**。

> **类比**：DNS 就像 **手机通讯录**——你不需要记住每个人的电话号码，只需要搜索名字就行。DNS 帮你把"名字"翻译成"号码"。

### 5.2 DNS 解析过程（步骤详解）

当你在浏览器输入 `www.example.com` 时：

1. **浏览器缓存**：先检查浏览器自己有没有记住这个域名的 IP（之前访问过的）
2. **操作系统缓存**：浏览器没有 → 问操作系统
3. **本地 DNS 服务器**（通常是你的 ISP 提供的）：操作系统也没有 → 去问 ISP 的 DNS 服务器
4. **Root DNS Server（根域名服务器）**：ISP 的 DNS 也不知道 → 问根服务器"谁管 .com？"
5. **TLD DNS Server（顶级域名服务器）**：根服务器说"去问 .com 的服务器" → 问 .com 服务器"谁管 example.com？"
6. **Authoritative DNS Server（权威域名服务器）**：.com 服务器说"去问 example.com 的服务器" → 得到最终的 IP 地址！
7. IP 地址沿原路返回给浏览器，浏览器用这个 IP 地址连接服务器

### 5.3 DNS 层级结构

```
          . (Root)
         / | \
      .com .org .cn    ← TLD (Top-Level Domain)
       |    |    |
  google  wiki  baidu  ← Second-Level Domain
     |
    www                ← Subdomain
```

In [ ]:
# 可视化：DNS 解析过程
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)
ax.set_title('DNS Resolution Process: How "www.example.com" becomes an IP Address',
             fontsize=13, fontweight='bold')
ax.axis('off')

# Components
components = {
    'browser': (1.5, 10, 'Browser', '#42A5F5'),
    'os':      (1.5, 7.5, 'OS Cache', '#66BB6A'),
    'local':   (5.5, 5.5, 'Local DNS\n(ISP)', '#FF7043'),
    'root':    (10, 9, 'Root DNS\nServer', '#EF5350'),
    'tld':     (10, 6.5, '.com TLD\nDNS Server', '#AB47BC'),
    'auth':    (10, 4, 'Authoritative\nDNS Server', '#26C6DA'),
    'web':     (10, 1.5, 'Web Server\n142.250.80.46', '#FF9800'),
}

for key, (x, y, label, color) in components.items():
    w, h = 2.2, 1.2
    rect = mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h, boxstyle="round,pad=0.15",
                                     facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# Arrows with step numbers
steps = [
    (1.5, 9.4, 1.5, 8.1, '1. Check\nbrowser cache', 'right'),
    (1.5, 6.9, 5.5, 6.1, '2. Check\nOS cache', 'left'),
    (5.5, 6.1, 10, 9.6, '3. Query\nRoot DNS', 'left'),
    (10, 8.4, 10, 7.1, '4. ".com is\nover there"', 'right'),
    (10, 5.9, 10, 4.6, '5. "example.com\nis over there"', 'right'),
    (10, 3.4, 5.5, 5.0, '6. Return IP:\n142.250.80.46', 'left'),
    (5.5, 4.9, 1.5, 7.0, '7. IP returned\nto browser', 'right'),
    (1.5, 9.4, 10, 2.1, '8. Connect to\nweb server!', 'left'),
]

arrow_colors = ['#1565C0', '#2E7D32', '#F4511E', '#B71C1C', '#6A1B9A', '#00838F', '#E65100', '#E65100']

for i, (x1, y1, x2, y2, label, side) in enumerate(steps):
    if i < 7:
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', color=arrow_colors[i], lw=2))
    else:
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', color='red', lw=2.5, ls='--'))

    mid_x = (x1 + x2) / 2
    mid_y = (y1 + y2) / 2
    offset = 0.8 if side == 'right' else -0.8
    ax.text(mid_x + offset, mid_y, label, fontsize=7, color=arrow_colors[min(i, 6)],
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='gray', alpha=0.9))

plt.tight_layout()
plt.savefig('dns_resolution.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. 网页是如何加载的？(How a Web Page Loads)

当你在浏览器输入 `https://www.example.com` 并按回车，背后发生了什么？

### 完整过程：

1. **DNS 解析**：浏览器通过 DNS 将域名解析为 IP 地址
2. **TCP 连接**：浏览器与 Web 服务器建立 TCP 连接（三次握手 Three-Way Handshake）
   - 浏览器 → 服务器：SYN（你好，我想连接）
   - 服务器 → 浏览器：SYN-ACK（好的，我准备好了）
   - 浏览器 → 服务器：ACK（收到，我们开始吧）
3. **HTTPS 加密**（如果是 HTTPS）：TLS/SSL 握手，建立加密通道
4. **HTTP 请求**：浏览器发送 HTTP GET 请求获取网页
5. **服务器处理**：Web 服务器找到请求的文件，生成响应
6. **HTTP 响应**：服务器返回 HTML 文件（状态码 200 OK）
7. **渲染页面**：
   - 浏览器解析 HTML，构建 DOM 树
   - 发现 CSS 文件 → 再发请求获取 CSS
   - 发现 JavaScript 文件 → 再发请求获取 JS
   - 发现图片 → 再发请求获取图片
   - 最终渲染出完整页面

> 这个过程通常在 **几百毫秒** 内完成！

In [ ]:
# 模拟网页加载过程的时间线
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.set_xlim(0, 1000)
ax.set_ylim(0, 12)
ax.set_title('Timeline: What Happens When You Load a Web Page', fontsize=13, fontweight='bold')
ax.set_xlabel('Time (milliseconds)', fontsize=11)
ax.axis('on')
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

steps_timeline = [
    (0, 50, 'DNS Lookup', '#42A5F5', 10.5),
    (50, 80, 'TCP Handshake', '#66BB6A', 9.5),
    (80, 130, 'TLS/SSL Handshake', '#AB47BC', 8.5),
    (130, 180, 'HTTP Request + Response\n(HTML)', '#FF7043', 7.5),
    (180, 250, 'Parse HTML + Request CSS', '#EF5350', 6.5),
    (250, 320, 'Parse CSS + Request JS', '#26C6DA', 5.5),
    (320, 400, 'Execute JavaScript', '#FF9800', 4.5),
    (180, 450, 'Download Images', '#78909C', 3.5),
    (400, 500, 'Render Page', '#4CAF50', 2.5),
    (500, 520, 'Page Complete!', '#F44336', 1.5),
]

for start, end, label, color, y in steps_timeline:
    width = end - start
    rect = mpatches.FancyBboxPatch((start, y-0.35), width, 0.7, boxstyle="round,pad=0.05",
                                     facecolor=color, edgecolor='black', linewidth=1, alpha=0.85)
    ax.add_patch(rect)
    text_x = start + width/2
    if width < 40:
        text_x = end + 5
        ax.text(text_x, y, label, fontsize=8, color=color, fontweight='bold', va='center')
    else:
        ax.text(text_x, y, label, fontsize=8, color='white', fontweight='bold',
                ha='center', va='center')
    ax.text(start, y+0.5, f'{start}ms', fontsize=7, color='gray', ha='center')

ax.axvline(x=500, color='red', linestyle='--', lw=1.5, alpha=0.5)
ax.text(510, 11, 'Page loaded!\n~500ms', fontsize=10, color='red', fontweight='bold')

ax.text(500, 0.5, 'All of this happens in about half a second!',
        fontsize=11, style='italic', color='#333', ha='center')

plt.tight_layout()
plt.savefig('page_load.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Ethernet 和 CSMA/CD

### 7.1 什么是 Ethernet？

**Ethernet（以太网）** 是目前最广泛使用的 **有线 LAN 技术**。它定义了数据如何在有线网络中传输。

**Ethernet 标准：**
| 标准 | 速度 | 线缆类型 |
|------|------|---------|
| 10BASE-T | 10 Mbps | Cat 3/5 |
| 100BASE-TX (Fast Ethernet) | 100 Mbps | Cat 5 |
| 1000BASE-T (Gigabit Ethernet) | 1 Gbps | Cat 5e/6 |
| 10GBASE-T | 10 Gbps | Cat 6a |

### 7.2 CSMA/CD (Carrier Sense Multiple Access with Collision Detection)

**CSMA/CD** 是 Ethernet 用来避免和处理 **数据碰撞** 的协议。

> **类比**：想象你在一个电话会议中——
> - **Carrier Sense（载波侦听）**：先听听有没有人在说话
> - **Multiple Access（多路访问）**：所有人都可以说话
> - **Collision Detection（碰撞检测）**：如果两个人同时说话（碰撞），都停下来

**CSMA/CD 工作步骤：**
1. **Listen（监听）**：设备在发送前先监听网络是否空闲
2. **Send（发送）**：如果空闲，开始发送数据
3. **Detect Collision（检测碰撞）**：发送的同时监听网络
4. **If Collision（如果碰撞）**：
   - 发送 **JAM 信号**，通知所有设备发生了碰撞
   - 所有涉及碰撞的设备停止发送
   - 每台设备等待一个 **随机时间** 后重试
   - 如果再次碰撞，等待时间加倍（指数退避 Exponential Backoff）

> **注意**：在现代 **全双工 Switch 网络** 中，CSMA/CD 基本不再需要了——因为 Switch 的每个端口是独立的碰撞域，数据不会碰撞。

In [ ]:
# 模拟 CSMA/CD 碰撞检测过程
import random

print("=" * 60)
print("  CSMA/CD Simulation")
print("=" * 60)

devices = ['PC-A', 'PC-B', 'PC-C']
max_attempts = 5

def csma_cd_simulate():
    # Simulate CSMA/CD for multiple devices trying to send
    round_num = 0
    pending = list(devices)
    completed = []

    while pending and round_num < 10:
        round_num += 1
        print(f"\n--- Round {round_num} ---")

        # Each pending device decides whether to try sending
        trying = []
        for dev in pending:
            # Carrier sense: check if channel is free
            if random.random() < 0.7:  # 70% chance of trying
                trying.append(dev)
                print(f"  {dev}: Channel seems free, attempting to send...")

        if len(trying) == 0:
            print("  No device attempted to send this round.")
        elif len(trying) == 1:
            # Success!
            dev = trying[0]
            print(f"  {dev}: Successfully sent data! No collision.")
            completed.append(dev)
            pending.remove(dev)
        else:
            # Collision!
            print(f"  COLLISION detected! {' and '.join(trying)} sent simultaneously!")
            print(f"  JAM signal sent to all devices.")
            for dev in trying:
                wait_time = random.randint(1, 2**min(round_num, 4))
                print(f"  {dev}: Will retry after waiting {wait_time} time slot(s)")

    print(f"\n{'=' * 60}")
    print(f"  Simulation complete!")
    print(f"  Successfully sent: {completed}")
    if pending:
        print(f"  Still waiting: {pending}")

random.seed(42)
csma_cd_simulate()

## 8. Cloud Computing（云计算）

### 8.1 什么是云计算？

**Cloud Computing** = 通过 Internet 按需访问计算资源（服务器、存储、软件等），不需要自己购买和维护硬件。

> **类比**：
> - **没有云计算** = 自己买发电机给家里供电（贵、要维护）
> - **有了云计算** = 从电网接电用（便宜、按用量付费、不用自己维护）

### 8.2 公有云 vs 私有云

| 特性 | Public Cloud (公有云) | Private Cloud (私有云) |
|------|---------------------|----------------------|
| 所有权 | 云服务商（如 AWS、Azure） | 组织自己 |
| 共享 | 多个客户共享基础设施 | 专属于一个组织 |
| 成本 | 按需付费，初始成本低 | 初始投资大 |
| 安全性 | 数据在外部服务器上 | 数据在自己的服务器上 |
| 灵活性 | 高（随时增减资源） | 有限 |
| 适合 | 中小企业、创业公司 | 银行、政府、医院 |

### 8.3 云服务模型

| 模型 | 含义 | 你管什么 | 例子 |
|------|------|---------|------|
| **IaaS** (Infrastructure as a Service) | 租基础设施 | 操作系统+应用 | AWS EC2、Azure VM |
| **PaaS** (Platform as a Service) | 租平台 | 只管应用代码 | Google App Engine |
| **SaaS** (Software as a Service) | 租软件 | 什么都不用管 | Gmail、Office 365 |

## 9. Bit Streaming（比特流）

### 什么是 Bit Streaming？

**Bit Streaming** = 数据（视频/音频）一边下载一边播放，不需要等到全部下载完。

### 两种类型：

#### Real-time Streaming（实时流）
- **直播**：数据在产生的同时就被传输和播放
- 例子：Twitch 直播、视频通话、在线课堂
- 特点：延迟要求低，不能暂停/回退

#### On-demand Streaming（点播流）
- **预先录制好的内容**：用户随时可以播放、暂停、快进、回退
- 例子：Netflix、B站、Spotify
- 特点：内容存储在服务器上，用户按需访问

### Streaming vs Download

| 特性 | Streaming | Download |
|------|-----------|----------|
| 何时可以开始 | 立即播放 | 全部下载后才能播放 |
| 存储需求 | 不占本地空间 | 需要足够的本地存储 |
| 网络需求 | 需要持续稳定的连接 | 下载完后可离线使用 |
| 对带宽要求 | 高（实时传输） | 灵活（可以慢慢下载） |

### Buffering（缓冲）

Streaming 时，系统会提前下载一小段数据存在 **缓冲区 (Buffer)** 中，这样即使网络短暂波动也不会中断播放。如果网络太慢，缓冲区的数据被消耗完，就会出现"加载中..."的转圈。

## 10. Internet 接入硬件

### 10.1 Modem（调制解调器）

**Modem** = **Mo**dulator + **Dem**odulator（调制 + 解调）

**功能**：把数字信号转换成可以在电话线/有线电视线上传输的模拟信号，以及反向转换。

> **类比**：Modem 就像 **翻译官**——你的电脑说"数字语言"，电话线说"模拟语言"，Modem 在两者之间翻译。

**类型：**
- **DSL Modem**：通过电话线（ADSL/VDSL）
- **Cable Modem**：通过有线电视线
- **Fibre Modem (ONT)**：通过光纤

### 10.2 PSTN (Public Switched Telephone Network)

- 传统的公共电话网络
- 早期 Internet 通过电话线拨号上网（Dial-up）
- 速度很慢（最大 56 Kbps）
- 上网时电话不能用（共享同一条线路）

### 10.3 Dedicated Lines（专用线路）

- 企业租用的专用通信线路
- 不与其他用户共享带宽
- 速度稳定、可靠性高
- 成本高
- 例子：T1 线路（1.5 Mbps）、T3 线路（45 Mbps）、光纤专线

### 10.4 接入方式对比

| 接入方式 | 速度 | 成本 | 特点 |
|---------|------|------|------|
| Dial-up (拨号) | 56 Kbps | 低 | 占用电话线，已淘汰 |
| ADSL | 24 Mbps下/3 Mbps上 | 中 | 非对称，上下行速度不同 |
| Cable (有线电视) | 1 Gbps | 中 | 与邻居共享带宽 |
| Fibre (光纤) | 10 Gbps+ | 高 | 最快最稳定 |
| 4G/5G (移动) | 1-20 Gbps | 中 | 无线，受信号影响 |
| Satellite (卫星) | 100 Mbps | 高 | 覆盖偏远地区，延迟高 |

In [ ]:
# 可视化：Internet 接入方式速度对比
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

access_types = [
    'Dial-up', 'ADSL', 'VDSL', 'Cable', '4G', '5G', 'Fibre\n(FTTH)', 'Satellite\n(LEO)'
]
download_speeds = [0.056, 24, 80, 1000, 300, 10000, 10000, 300]  # Mbps
colors_access = ['#9E9E9E', '#FF9800', '#FF7043', '#42A5F5', '#66BB6A', '#4CAF50', '#2196F3', '#AB47BC']

bars = ax.bar(access_types, download_speeds, color=colors_access, edgecolor='black', linewidth=1)
ax.set_ylabel('Download Speed (Mbps)', fontsize=11)
ax.set_title('Internet Access Methods: Speed Comparison', fontsize=14, fontweight='bold')
ax.set_yscale('log')

for bar, speed in zip(bars, download_speeds):
    if speed >= 1000:
        label = f'{speed/1000:.0f} Gbps'
    else:
        label = f'{speed} Mbps'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.3,
            label, ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(y=25, color='red', linestyle='--', lw=1, alpha=0.7)
ax.text(7.5, 30, 'FCC broadband threshold (25 Mbps)', fontsize=8, color='red', style='italic')

plt.tight_layout()
plt.savefig('access_speeds.png', dpi=100, bbox_inches='tight')
plt.show()

# Modem 的工作原理说明
print("=" * 55)
print("  Modem: Modulation & Demodulation")
print("=" * 55)
print()
print("  Digital Signal (Computer)")
print("  1  0  1  1  0  1  0  0")
print("  |  |  |  |  |  |  |  |")
print("  v  v  v  v  v  v  v  v")
print("  [    MODULATOR    ]  ← Modem converts to analog")
print("  |  |  |  |  |  |  |  |")
print("  ~~~~~~~~~~~~~~~~~~~~~ ← Analog signal on phone line")
print("  |  |  |  |  |  |  |  |")
print("  [ DEMODULATOR     ]  ← Modem converts back to digital")
print("  |  |  |  |  |  |  |  |")
print("  1  0  1  1  0  1  0  0")
print("  Digital Signal (Remote Computer)")

## 11. 练习题 (Practice Exercises)

### 选择题

**Q1.** Internet 和 WWW 的关系是？
- A) 它们是同一个东西
- B) Internet 是 WWW 的子集
- C) WWW 是运行在 Internet 上的一个服务
- D) WWW 比 Internet 更早出现

**Q2.** IPv4 地址由多少个 bits 组成？
- A) 16
- B) 32
- C) 64
- D) 128

**Q3.** DNS 的主要功能是什么？
- A) 加密网络数据
- B) 将域名解析为 IP 地址
- C) 过滤网络流量
- D) 分配 IP 地址

**Q4.** 在 CSMA/CD 中，当检测到碰撞时，设备会：
- A) 立即重新发送数据
- B) 发送 JAM 信号然后等待随机时间后重试
- C) 永久停止发送
- D) 切换到另一条线路

**Q5.** 以下哪个是 Public Cloud 的优势？
- A) 数据存储在自己的服务器上
- B) 按需付费，初始成本低
- C) 不需要 Internet 连接
- D) 不与其他客户共享资源

**Q6.** Modem 的功能是什么？
- A) 连接不同的网络并选择路径
- B) 在数字信号和模拟信号之间转换
- C) 放大网络信号
- D) 分配 IP 地址

---

### 参考答案
1. **C** — WWW 是 Internet 上的一个应用/服务
2. **B** — IPv4 是 32 bits (4 × 8)
3. **B** — DNS 将域名翻译为 IP 地址
4. **B** — 碰撞后发送 JAM 信号，然后随机等待重试
5. **B** — 公有云的主要优势是按需付费和弹性扩展
6. **B** — Modem = Modulator + Demodulator

### 简答题

**Q7.** 描述当用户在浏览器中输入 `https://www.example.com` 并按回车后，直到网页显示出来的完整过程。（6分）

<details>
<summary>点击查看参考答案</summary>

1. 浏览器通过 **DNS** 将 `www.example.com` 解析为 IP 地址（1分）
2. 浏览器与 Web 服务器建立 **TCP 连接**（三次握手）（1分）
3. 由于是 HTTPS，进行 **TLS/SSL 握手** 建立加密连接（1分）
4. 浏览器发送 **HTTP GET 请求** 获取网页（1分）
5. 服务器处理请求并返回 **HTML 响应**（1分）
6. 浏览器解析 HTML，**下载额外资源**（CSS、JavaScript、图片等），最终 **渲染** 出完整页面（1分）

</details>

**Q8.** 解释为什么需要从 IPv4 过渡到 IPv6。（3分）

<details>
<summary>点击查看参考答案</summary>

- IPv4 只有 **32 bits**，最多支持约 43 亿个地址（1分）
- 随着联网设备数量爆炸性增长（手机、IoT设备等），IPv4 地址已经 **不够用** 了（1分）
- IPv6 使用 **128 bits**，提供几乎无限的地址空间（3.4×10^38），同时还改进了安全性和路由效率（1分）

</details>

**Q9.** 比较 Real-time Streaming 和 On-demand Streaming，各举一个例子。（4分）

<details>
<summary>点击查看参考答案</summary>

**Real-time Streaming（实时流）：**
- 数据在产生的 **同时** 被传输和播放（1分）
- 例子：Twitch 游戏直播、视频会议（1分）

**On-demand Streaming（点播流）：**
- 内容 **预先录制** 并存储在服务器上，用户随时可以播放、暂停、回退（1分）
- 例子：Netflix 看电影、B站看视频（1分）

</details>

---
*第2章 Section 2.1 完成！你已经学习了网络通信的核心概念。继续加油！*